<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/944_EAASv3_DataLoading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EaaS v3 Data Loading Layer — Architecture Review

## Why Data Loading Matters in an Evaluation System

Before an evaluation engine can measure performance or detect risk, it must answer three foundational questions:

1. **Where is the evaluation data located?**
2. **Which run is being evaluated?**
3. **What historical context should be used for comparison?**

This module is responsible for answering those questions.

Rather than scattering file loading logic throughout the orchestrator nodes, this code centralizes all data ingestion inside a **dedicated utility module**. This makes the system:

* easier to audit
* easier to debug
* easier to reuse in future agents

Operationally, this module transforms **raw JSON files** into a **structured evaluation context** stored in the agent’s state.

---

# Role in the Overall Agent Workflow

Within the orchestrator graph, this module powers the **first node in the system**:

```
load_data_node
      ↓
score_run_node
      ↓
update_trigger_history_node
      ↓
report_node
```

Its job is to populate the state with everything the scoring engine needs:

* evaluation scenarios
* scenario metadata
* trigger history
* selected run
* previous run metrics

Once this step finishes, the agent has a **complete and deterministic snapshot of the system under evaluation**.

---

# Project Root Resolution

### `_resolve_project_root`

```python
def _resolve_project_root(state: EvalAsServiceState) -> Path
```

This function determines **where the project is located on disk**.

The system checks two possibilities:

1️⃣ If `project_root` was already provided in the state
2️⃣ Otherwise infer it from the current file location

This fallback strategy makes the agent much more portable.

It allows the orchestrator to run correctly whether it is executed:

* from a CLI entry point
* inside a notebook
* from a test harness
* from an orchestrator registry

For operational systems, this kind of path resolution prevents a large class of deployment errors.

---

# Safe JSON Loading

### `_load_json`

```python
def _load_json(path: Path) -> List[Dict[str, Any]]
```

This utility loads JSON data while **protecting the system from malformed inputs**.

Key safety features:

• Returns an empty list if the file does not exist
• Ensures the result is a list (the expected structure)
• Avoids runtime crashes during evaluation

This is a subtle but important design decision.

Evaluation systems should **fail gracefully**, because evaluation failures should not disrupt the system being evaluated.

---

# Automatic Run Selection

### `_select_run_id`

```python
def _select_run_id(
    scenario_results_history,
    explicit_run_id
)
```

This function determines **which evaluation run should be analyzed**.

The logic follows a clear hierarchy:

1️⃣ If the user explicitly provided a `run_id`, use it
2️⃣ Otherwise select the **most recent run** from the results history

The fallback behavior assumes that the evaluation history is **append-only**, meaning the latest entry represents the newest run.

This allows operators to run the agent without specifying a run ID every time.

Operationally, that makes the system behave like a monitoring service rather than a manual analysis tool.

---

# Scenario Filtering

### `_filter_scenarios_for_run`

```python
def _filter_scenarios_for_run(
    scenario_results_history,
    run_id,
    target_version
)
```

This function extracts **the subset of scenarios belonging to the run being evaluated**.

It filters based on two criteria:

| Filter         | Purpose                                             |
| -------------- | --------------------------------------------------- |
| run_id         | selects the evaluation batch                        |
| target_version | optionally isolates a specific orchestrator version |

This separation is extremely useful during system development.

For example:

* evaluating a new release candidate
* comparing two versions of the orchestrator
* isolating regression tests

Instead of modifying datasets, the evaluator can simply filter the run.

That keeps evaluation **reproducible and transparent**.

---

# Historical Context Detection

### `_find_previous_run_metrics_from_trigger_history`

```python
def _find_previous_run_metrics_from_trigger_history(
    trigger_history,
    current_run_id
)
```

This function determines **what the previous evaluation run was**.

It searches the trigger history for the entry associated with the current run.

Two scenarios are handled:

**Case 1 — Run already recorded**

If the current run exists in the history, the function returns the run immediately before it.

**Case 2 — New run**

If the run is not yet in history, the function simply returns the latest entry.

This allows the system to perform **trend analysis** without requiring perfect historical alignment.

It ensures that the scoring engine always has some baseline for comparison.

---

# Main Data Loading Pipeline

### `load_eaas_data`

```python
def load_eaas_data(
    state,
    config
)
```

This function orchestrates the entire data ingestion process.

It performs the following steps:

### 1️⃣ Resolve project root

Ensures all paths resolve correctly across environments.

---

### 2️⃣ Build data paths

The system constructs paths to:

```
scenario_results_history.json
trigger_history.json
scenario_catalog_enriched.json
```

These datasets form the **evaluation backbone** of EaaS.

---

### 3️⃣ Load evaluation datasets

Each file is loaded into the state:

```
scenario_results_history
trigger_history
scenario_catalog
```

This keeps the raw data accessible to downstream nodes.

---

### 4️⃣ Select evaluation run

The system determines the run being analyzed.

```
state["run_id"]
```

This step ensures the agent evaluates a **single, well-defined system snapshot**.

---

### 5️⃣ Extract run-specific scenarios

The system isolates the scenarios belonging to the selected run.

```
state["scenario_results_for_run"]
```

This dataset is what the scoring engine will evaluate.

---

### 6️⃣ Determine previous run metrics

The agent identifies the metrics of the prior run.

```
state["previous_run_metrics"]
```

This enables trend detection such as:

* regressions
* improvements
* stability changes

---

# Why This Design Is Strong

Several design decisions here significantly improve reliability.

---

## Deterministic Data Selection

The system does not rely on AI to decide what data to evaluate.

Instead it follows explicit rules:

* run selection
* version filtering
* dataset slicing

This makes the evaluation **fully reproducible**.

---

## Clean Separation of Concerns

The module performs only one task:

> **Loading and preparing evaluation data**

It does not:

* compute scores
* generate reports
* detect triggers

Those responsibilities are delegated to other modules.

This separation keeps the architecture modular and testable.

---

## Graceful Failure Handling

Missing files do not crash the system.

Instead they return empty lists.

That behavior is critical for real operational systems where evaluation data may not always exist yet.

---

# Suggested Improvements

The implementation is already strong, but a few refinements could improve robustness.

---

# 1. Validate Scenario Catalog Consistency

Right now the module loads the catalog but does not verify it.

You may want to add a quick validation step:

```
ensure scenario_id exists
ensure severity_weight exists
ensure business_risk exists
```

This protects the scoring engine from malformed catalog entries.

---

# 2. Warn When No Scenarios Exist for a Run

If this happens:

```
scenario_results_for_run = []
```

the system should log an error.

Otherwise the evaluation engine may produce misleading results.

---

# 3. Detect Multiple Target Versions

If `target_version` filtering is used, you may want to confirm that all scenarios belong to the same orchestrator version.

Mixed versions could invalidate evaluation results.

---

# Final Assessment

This module is **well structured, production-appropriate, and aligned with your architectural philosophy**.

It demonstrates several strong engineering practices:

* deterministic data loading
* safe JSON handling
* explicit run selection
* historical context detection
* modular separation from scoring logic

Most importantly, it reinforces the central principle behind EaaS:

> **AI evaluation should be transparent, reproducible, and governed by explicit rules.**

This data-loading layer ensures the evaluation engine always operates on **a clearly defined system snapshot**, which is essential for producing trustworthy evaluation results.




In [ ]:
"""Data loading utilities for EaaS v3.

Responsible for:
- resolving project_root
- loading scenario_results_history, trigger_history, scenario_catalog
- selecting a run_id (if not provided)
- deriving scenario_results_for_run and previous_run_metrics
"""

from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import json

from config import EvalAsServiceState, EvalAsServiceConfig


def _resolve_project_root(state: EvalAsServiceState) -> Path:
    """Best-effort resolution of project root."""
    if state.get("project_root"):
        return Path(state["project_root"]).resolve()
    # Fallback: assume this file is under project_root/agents/...
    return Path(__file__).resolve().parents[4]


def _load_json(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        return data
    return []


def _select_run_id(
    scenario_results_history: List[Dict[str, Any]],
    explicit_run_id: Optional[str],
) -> Optional[str]:
    if explicit_run_id:
        return explicit_run_id
    # Fallback: latest run_id by appearance order
    if not scenario_results_history:
        return None
    # Preserve order and select last run_id
    last = scenario_results_history[-1]
    return last.get("run_id")


def _filter_scenarios_for_run(
    scenario_results_history: List[Dict[str, Any]],
    run_id: Optional[str],
    target_version: Optional[str],
) -> List[Dict[str, Any]]:
    if not run_id:
        return []
    results: List[Dict[str, Any]] = []
    for row in scenario_results_history:
        if row.get("run_id") != run_id:
            continue
        if target_version and row.get("target_version") != target_version:
            continue
        results.append(row)
    return results


def _find_previous_run_metrics_from_trigger_history(
    trigger_history: List[Dict[str, Any]],
    current_run_id: Optional[str],
) -> Optional[Dict[str, Any]]:
    if not trigger_history or not current_run_id:
        return None
    # Find index of current run in trigger history (if present)
    idx = None
    for i, entry in enumerate(trigger_history):
        if entry.get("run_id") == current_run_id:
            idx = i
            break
    if idx is None:
        # Not in history yet; use last entry as previous run if exists
        if len(trigger_history) >= 1:
            return trigger_history[-1]
        return None
    if idx == 0:
        return None
    return trigger_history[idx - 1]


def load_eaas_data(
    *,
    state: EvalAsServiceState,
    config: EvalAsServiceConfig,
) -> EvalAsServiceState:
    """Load all EaaS data into state and derive run-specific slices."""
    project_root = _resolve_project_root(state)
    state["project_root"] = str(project_root)

    data_dir = state.get("data_dir") or config.data_dir
    data_root = project_root / data_dir

    scenario_results_path = data_root / config.scenario_results_history_file
    trigger_history_path = data_root / config.trigger_history_file
    scenario_catalog_path = data_root / config.scenario_catalog_file

    scenario_results_history = _load_json(scenario_results_path)
    trigger_history = _load_json(trigger_history_path)
    scenario_catalog = _load_json(scenario_catalog_path)

    state["scenario_results_history"] = scenario_results_history
    state["trigger_history"] = trigger_history
    state["scenario_catalog"] = scenario_catalog

    run_id = _select_run_id(scenario_results_history, state.get("run_id"))
    state["run_id"] = run_id

    scenario_results_for_run = _filter_scenarios_for_run(
        scenario_results_history=scenario_results_history,
        run_id=run_id,
        target_version=state.get("target_version"),
    )
    state["scenario_results_for_run"] = scenario_results_for_run

    previous_run_metrics = _find_previous_run_metrics_from_trigger_history(
        trigger_history=trigger_history,
        current_run_id=run_id,
    )
    state["previous_run_metrics"] = previous_run_metrics

    return state

